# Loading the model

In [8]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

In [7]:
import transformers
print(transformers.__version__)
print(transformers.__file__)
print(hasattr(transformers, "T5ForConditionalGeneration"))

5.3.0
C:\Users\DELL\anaconda3\Lib\site-packages\transformers\__init__.py
True


In [9]:
model=T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer=T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

# Test the core logic for summarization

In [17]:
from torch.cuda import is_available
import torch

if torch.cuda.is_available():
  device = torch.device("cuda")
  print(device)
else:
  device = torch.device("cpu")
  print(device)


cpu


In [28]:
import re
def clean_data(text):
  text = re.sub(r'\r\n', ' ', text)
  text = re.sub(r'\s+', ' ', text)
  text = re.sub(r'<.*?>', ' ', text)
  text = text.strip().lower()
  return text
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # clean

    # tokenise
    inputs = tokenizer(dialogue, padding="max_length", max_length=512, truncation=True, return_tensors="pt")
    
    # Generate the summary => token ids
    model.to(device)
    targets = model.generate(
        input_ids=inputs["input_ids"], 
        attention_mask=inputs["attention_mask"], 
        max_length=150, 
        num_beams=4, # it will create 4 different summaries and give the best out of it
        early_stopping=True
    )
    
    # token ids convert to summary => decoding
    summary = tokenizer.decode(targets[0], skip_special_tokens=True) # EOS, SEP
    
    return summary

In [29]:
# sample conversation
test_dialogue = """
Reporter: Good morning, and welcome. Today, we are discussing the rapidly shifting landscape of the workforce. Joining us is Dr. Aris Vance, a labor economist specializing in technology trends. Dr. Vance, thank you for being here. Is AI truly coming for everyone’s jobs, or is that an exaggeration?
Expert: Thank you for having me. To put it bluntly: it is an exaggeration, but the anxiety is justified. AI will not completely eliminate most jobs, but it will fundamentally change how almost every job is done. We are looking at a shift from replacement to augmentation.
Reporter: Walk us through that distinction. What does augmentation look like for an average office worker?
Expert: Think of AI as a highly capable intern. If you are a graphic designer, AI can generate ten rough concepts in seconds. Your job shifts from drawing those initial drafts by hand to acting as the creative director—curating, refining, and applying human emotional intelligence to the final product. The time spent on tedious tasks shrinks, while time spent on critical thinking expands.
Reporter: That sounds positive, but there must be a downside. Which industries are facing the highest risk of actual displacement?
Expert: The data points heavily toward roles centered on routine information processing. Data entry clerks, basic customer support roles, and compliance analysts are seeing the highest rates of automation. If a job involves following a strict, predictable rulebook to process data, an AI can now do it faster and cheaper.
Reporter: For workers in those high-risk fields, what is the most urgent step they should take right now?
Expert: Upskilling is the only path forward. Workers need to focus on what I call "human-exclusive skills." AI lacks empathy, ethical judgment, and complex negotiation skills. Moving toward roles that require deep human connection, leadership, or cross-disciplinary problem-solving is the best way to remain indispensable.
Reporter: Dr. Vance, thank you for breaking down a complex issue into actionable insights.
Expert: My pleasure. Thank you.
"""
test_summary = summarize_dialogue(test_dialogue)

In [30]:
print(test_summary)

ai is joining dr. aris vance, a labor economist specializing in technology trends. ai will not eliminate most jobs, but fundamentally change how almost every job is done. ai is a highly capable intern and can generate ten rough concepts in seconds. ai lacks empathy, ethical judgment and complex negotiation skills.
